# Image Clustering Using VGG16 and K-Means by Mateo Vergara

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Importing libraries, loading a pre-trained ResNet50 CNN, and defining functions for preprocessing and feature extraction

In [ ]:
import os
import shutil
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.preprocessing import image
from keras.models import Model

def extract_features(folder_path, model, input_shape=(224, 224)): #Feature extraction function and changing image size to 224x224
    features = []
    file_paths = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff")):
            file_path = os.path.join(folder_path, filename)
            try:
                img = image.load_img(file_path, target_size=input_shape)
                img_data = image.img_to_array(img)
                img_data = np.expand_dims(img_data, axis=0)
                img_data = preprocess_input(img_data)
                feature = model.predict(img_data)
                features.append(feature.flatten())
                file_paths.append(file_path)

            except Exception as e:
                print(f"Error processing {file_path}: {e}")

    return np.array(features), file_paths


## Clustering related functions and main execution

In [ ]:
def cluster_images(features, n_clusters):

    kmeans = KMeans(n_clusters = n_clusters, random_state=0) #We're using KMeans for clustering
    labels = kmeans.fit_predict(features)
    return labels

def save_clustered_images(labels, file_paths, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    for label in set(labels):
        cluster_folder = os.path.join(output_folder, f"cluster_{label}")
        os.makedirs(cluster_folder, exist_ok=True)

        for i, file_path in enumerate(file_paths):
            if labels[i] == label:
                shutil.copy(file_path, cluster_folder)
                print(f"Image {file_path} saved to {cluster_folder}")

def find_and_cluster_images(input_folder, output_folder, n_clusters):

    base_model = ResNet50(weights='imagenet') #Loading pre-trained ResNet50 CNN
    model = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

    features, file_paths = extract_features(input_folder, model)

    if len(file_paths) == 0:
        print("No images found in the folder.")
        return

    labels = cluster_images(features, n_clusters=n_clusters)

    save_clustered_images(labels, file_paths, output_folder)


input_folder = '/content/your_directory' #Change this path to your input folder
output_folder = '/content/output_folder' #Your output folder, leave it as it is unless necessary

find_and_cluster_images(input_folder, output_folder, n_clusters = 3) #Change the number of clusters to yout needs

## Compressing the clustered data and downloading it

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/output_folder", 'zip', output_folder) #Keep this line as it is if you haven't change the output_folder in block 6

files.download("/content/output_folder.zip") #The same path as the output folder but adding the .zip extension


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>